# Tóm tắt các bước xử lý dữ liệu shop

Notebook này thực hiện các bước sau đối với dữ liệu shop:


1. **Đọc dữ liệu gốc** từ file `shop_original.csv` trong thư mục `preprocessed`.

2. **Chuyển đổi các cột phần trăm** (ví dụ: Đánh giá người bán, Giao đúng hẹn, Tỷ lệ hủy đơn, Tỷ lệ phản hồi chat) từ dạng chuỗi có ký tự `%` sang số thực (float).

3. **Chuyển đổi cột 'Đã tham gia'** từ dạng chuỗi (ví dụ: '5 + năm', '9 + tháng') sang số năm (float).

4. **Làm tròn giá trị số năm tham gia** đến 4 chữ số thập phân.

5. **Bổ sung cột 'Tên shop'** vào file `shop_cleaned.csv` dựa trên dữ liệu từ `sanpham_cleaned.csv`.

6. **Thay thế giá trị null trong cột 'Tên shop'** bằng 'N/A'.

7. **Lưu dữ liệu đã xử lý** ra file `shop_cleaned.csv` trong cùng thư mục.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

d:\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\Anaconda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [9]:
# --- Path Setup ---
# Đường dẫn tuyệt đối tới file shop_original.csv
shop_path = Path(r'd:/HCMUS/HOCTAP/DataVisualization/Lab/test/preprocessed/shop_original.csv')
pre_dir = shop_path.parent

# --- Helper Functions ---
def parse_percentage(value):
    """Chuan hoa gia tri phan tram (vi du: '93%') thanh float (vi du: 0.93)."""
    if pd.isna(value):
        return np.nan
    text = str(value).replace('%', '').strip()
    if not text:
        return np.nan
    try:
        return float(text) / 100.0
    except (ValueError, TypeError):
        return np.nan

def parse_joined_date_to_years(value):
    """Chuan hoa 'Đã tham gia' (vi du: '5 + năm', '9 + tháng') thanh so nam (float)."""
    if pd.isna(value):
        return np.nan
    text = str(value).lower().strip()
    
    # Tim so nam
    match_year = re.search(r'(\d+)\s*\+\s*năm', text)
    if match_year:
        return float(match_year.group(1))
        
    # Tim so thang
    match_month = re.search(r'(\d+)\s*\+\s*tháng', text)
    if match_month:
        return float(match_month.group(1)) / 12.0
        
    return np.nan

# --- Main Processing ---
if not shop_path.exists():
    raise FileNotFoundError(f"File not found: {shop_path}")

df = pd.read_csv(shop_path)
print("Kiểu dữ liệu gốc:")
print(df.info())

# --- YÊU CẦU MỚI ---
# 1. Cột "Seller ID" nếu null thì viết 'N/A', sau đó chuyển thành chuỗi
if 'Seller ID' in df.columns:
    df['Seller ID'] = df['Seller ID'].fillna('N/A').astype(str)

# 2. Các cột số nếu NaN thì chuyển thành None (để lưu vào MongoDB là null)
null_cols = [
    'Followers',
    'Đánh giá người bán',
    'Số năm tham gia',
    'Giao đúng hẹn',
    'Tỷ lệ hủy đơn',
    'Tỷ lệ phản hồi chat'
    ]
for col in null_cols:
    if col in df.columns:
        df[col] = df[col].where(~df[col].isna(), None)

# 3. Chuẩn hóa các cột phần trăm trước
percentage_cols = [
    'Đánh giá người bán', 
    'Giao đúng hẹn', 
    'Tỷ lệ hủy đơn', 
    'Tỷ lệ phản hồi chat'
    ]
for col in percentage_cols:
    if col in df.columns:
        df[col] = df[col].apply(parse_percentage)

# 4. Chuẩn hóa cột 'Đã tham gia'
if 'Đã tham gia' in df.columns:
    df['Đã tham gia'] = df['Đã tham gia'].apply(parse_joined_date_to_years)
    df = df.rename(columns={'Đã tham gia': 'Số năm tham gia'})

# 5. Ép kiểu float và làm tròn 4 chữ số thập phân cho các cột số
float_cols = [
    'Đánh giá người bán',
    'Số năm tham gia',
    'Giao đúng hẹn',
    'Tỷ lệ hủy đơn',
    'Tỷ lệ phản hồi chat'
    ]
for col in float_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(float).round(4)

# --- Save and Display Results ---
save_path = pre_dir / 'shop_cleaned.csv'
try:
    df.to_csv(save_path, index=False, encoding='utf-8-sig')
except PermissionError:
    print(f'\nFile {save_path.name} đang bị khóa, không thể ghi đè.')

print(f'\nĐã cập nhật file: {save_path}')
print("\nKiểu dữ liệu sau khi chuẩn hóa:")
print(df.info())

print("\n5 dòng đầu tiên sau khi xử lý:")
display(df.head())

Kiểu dữ liệu gốc:
<class 'pandas.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Shop URL             1980 non-null   str    
 1   Seller ID            1952 non-null   float64
 2   Loại cửa hàng        1980 non-null   str    
 3   Followers            1975 non-null   float64
 4   Đánh giá người bán   1960 non-null   str    
 5   Danh mục chính       1980 non-null   str    
 6   Đã tham gia          1683 non-null   str    
 7   Giao đúng hẹn        1764 non-null   str    
 8   Tỷ lệ hủy đơn        378 non-null    str    
 9   Tỷ lệ phản hồi chat  1320 non-null   str    
dtypes: float64(2), str(8)
memory usage: 385.6 KB
None

Đã cập nhật file: d:\HCMUS\HOCTAP\DataVisualization\Lab\test\preprocessed\shop_cleaned.csv

Kiểu dữ liệu sau khi chuẩn hóa:
<class 'pandas.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 10 columns):
 #   Colum

,Shop URL,Seller ID,Loại cửa hàng,Followers,Đánh giá người bán,Danh mục chính,Số năm tham gia,Giao đúng hẹn,Tỷ lệ hủy đơn,Tỷ lệ phản hồi chat
0,https://www.lazada.vn/shop/oqrz1tvi/?path=prof...,200983952195.0,Shop Thường,55.0,0.93,Beauty,0.75,0.98,NaN,0.8333
1,https://www.lazada.vn/shop/laikou-selection/?p...,200548320478.0,Shop Thường,201.0,0.98,Beauty,2.00,1.00,NaN,NaN
2,https://www.lazada.vn/shop/on-moon/?path=profi...,200164386065.0,Shop Thường,18894.0,0.96,Beauty,5.00,0.97,0.03,0.2000
3,https://www.lazada.vn/shop/ddddvn-selection/?p...,200522832214.0,Shop Thường,1505.0,0.98,Kitchenware & Tableware,2.00,0.98,NaN,1.0000
4,https://www.lazada.vn/shop/shopdep24h-chuyen-h...,1000442453.0,Shop Thường,21377.0,0.99,Beauty,6.00,1.00,NaN,0.5000


In [12]:
# Đọc dữ liệu
df_shop = pd.read_csv('../preprocessed/shop_cleaned.csv', dtype={'Seller ID': str})
df_sp = pd.read_csv('../preprocessed/sanpham_cleaned.csv', dtype={'Seller ID': str})

# Lấy cặp Seller ID - Tên shop
shop_names = df_sp.groupby('Seller ID', as_index=False)['Tên shop'].first()
df_shop = df_shop.merge(shop_names, on='Seller ID', how='left')

# Lưu lại
df_shop.to_csv('../preprocessed/shop_cleaned.csv', index=False, encoding='utf-8-sig')
print('Đã bổ sung cột Tên shop vào shop_cleaned.csv')

Đã bổ sung cột Tên shop vào shop_cleaned.csv


In [17]:
# Nếu cột "Tên shop" là null thì đổi thành "N/A"
if 'Tên shop' in df_shop.columns:
    df_shop['Tên shop'] = df_shop['Tên shop'].fillna('N/A')
    # Nếu muốn lưu lại ngay:
    df_shop.to_csv('../preprocessed/shop_cleaned.csv', index=False, encoding='utf-8-sig')
    print('Đã thay thế giá trị null trong cột Tên shop bằng "N/A"')
else:
    print('Không tìm thấy cột "Tên shop" trong df_shop')

Đã thay thế giá trị null trong cột Tên shop bằng "N/A"


In [19]:
# Liệt kê kiểu dữ liệu của các cột trong file shop_cleaned.csv
shop_cleaned_path = pre_dir / 'shop_cleaned.csv'

if shop_cleaned_path.exists():
    df_info = pd.read_csv(shop_cleaned_path, dtype={'Seller ID': str})
    print(f"Kiểu dữ liệu của các cột trong file: {shop_cleaned_path.name}\n")
    df_info.info()
else:
    # Fallback to shop_original.csv if shop_cleaned.csv doesn't exist
    shop_path = pre_dir / 'shop_original.csv'
    if shop_path.exists():
        df_info = pd.read_csv(shop_path, dtype={'Seller ID': str})
        print(f"Kiểu dữ liệu của các cột trong file: {shop_path.name}\n")
        df_info.info()
    else:
        print(f"Không tìm thấy file shop_cleaned.csv hoặc shop_original.csv")

Kiểu dữ liệu của các cột trong file: shop_cleaned.csv

<class 'pandas.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Shop URL             1980 non-null   str    
 1   Seller ID            1952 non-null   str    
 2   Loại cửa hàng        1980 non-null   str    
 3   Followers            1975 non-null   float64
 4   Đánh giá người bán   1960 non-null   float64
 5   Danh mục chính       1980 non-null   str    
 6   Số năm tham gia      1683 non-null   float64
 7   Giao đúng hẹn        1764 non-null   float64
 8   Tỷ lệ hủy đơn        378 non-null    float64
 9   Tỷ lệ phản hồi chat  1320 non-null   float64
 10  Tên shop             1913 non-null   str    
dtypes: float64(6), str(5)
memory usage: 424.4 KB
